[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/47_cutmix_solution.ipynb)

# Solution: CutMix

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def cutmix(x, y, alpha=1.0):
    B, C, H, W = x.shape
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    perm = torch.randperm(B, device=x.device)
    cut_ratio = (1 - lam) ** 0.5
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cy = torch.randint(0, H, (1,)).item()
    cx = torch.randint(0, W, (1,)).item()
    y1 = max(0, cy - cut_h // 2)
    y2 = min(H, cy + cut_h // 2)
    x1 = max(0, cx - cut_w // 2)
    x2 = min(W, cx + cut_w // 2)
    x_mixed = x.clone()
    x_mixed[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    # Recompute lam from realized area
    lam = 1.0 - (y2 - y1) * (x2 - x1) / (H * W)
    return x_mixed, y, y[perm], lam

In [ ]:
import torch
torch.manual_seed(0)
x = torch.zeros(2, 1, 16, 16)
x[1] = 1.0
y = torch.tensor([0, 1])
x_mix, y_a, y_b, lam = cutmix(x, y, alpha=1.0)
print('lam =', round(lam, 3))
print('Ones in sample 0:', (x_mix[0] == 1).sum().item(), '(should be (1-lam)*256 if perm swapped)')

In [ ]:
from torch_judge import check
check("cutmix")